# PPO Optimizer-Chain Evaluation Notebook

This notebook loads a saved PPO model and evaluates it on either the validation or test split. Set `EVAL_SPLIT` to choose which split to evaluate. The environment is imported from `optimizer_chain_env.py` so prediction uses the same stepping logic as training.


In [ ]:
from pathlib import Path
import os
import sys
import warnings

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from stable_baselines3 import PPO

RANDOM_STATE = 42
RUNTIME_WEIGHT = 0.1
MODEL_PATH = Path("optimizer_chain_ppo_0.1.zip")
EVAL_SPLIT = "test"

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "dbs" / "original_data" / "one_and_two_opt_master.csv").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

CHAIN_OPTIMIZER_DIR = PROJECT_ROOT / "machine_learning" / "RL" / "two_opt_chain_rl_v1"
sys.path.insert(0, str(CHAIN_OPTIMIZER_DIR))
from optimizer_chain_env import METRIC_NAMES, OptimizerChainEnv

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)


## 1. Load Data and Recreate Train / Validation / Test Split

In [ ]:
CSV_PATH = PROJECT_ROOT / "dbs" / "original_data" / "one_and_two_opt_master.csv"

master_df = pd.read_csv(CSV_PATH)

required_columns = [
    "original_circuit",
    "original_category",
    "chain_name",
    "initial_num_qubits",
    "initial_depth",
    "initial_two_qubit_gates",
    "initial_total_gates",
    "initial_two_qubit_depth",
    "initial_gate_density",
    "initial_two_qubit_ratio",
    "final_num_qubits",
    "final_depth",
    "final_two_qubit_gates",
    "final_total_gates",
    "final_two_qubit_depth",
    "total_runtime_seconds",
]

missing = sorted(set(required_columns) - set(master_df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = master_df.copy()
df["two_qubit_gate_reduction"] = (
    df["initial_two_qubit_gates"] - df["final_two_qubit_gates"]
)
df["two_qubit_gate_reduction_pct"] = (
    100
    * df["two_qubit_gate_reduction"]
    / df["initial_two_qubit_gates"].replace(0, np.nan)
)

df = df.dropna(
    subset=[
        "two_qubit_gate_reduction_pct",
        "total_runtime_seconds",
        "final_two_qubit_gates",
    ]
).copy()

OPTIMIZER_NAMES = [
    "qiskit_standard",
    "qiskit_ai",
    "tket",
    "wisq_rules",
    "wisq_bqskit",
]
STOP_ACTION = len(OPTIMIZER_NAMES)
CATEGORY_NAMES = sorted(df["original_category"].unique())

summary = pd.DataFrame(
    {
        "rows": [len(df)],
        "original_circuits": [df["original_circuit"].nunique()],
        "chains": [df["chain_name"].nunique()],
        "categories": [df["original_category"].nunique()],
        "error_true_rows": [
            int(df.get("error", pd.Series(False, index=df.index)).astype(bool).sum())
        ],
    }
)
summary


In [ ]:
circuits = np.array(sorted(df["original_circuit"].unique()))

train_val_circuits, test_circuits = train_test_split(
    circuits,
    test_size=0.10,
    random_state=RANDOM_STATE,
)
train_circuits, val_circuits = train_test_split(
    train_val_circuits,
    test_size=0.10 / 0.90,
    random_state=RANDOM_STATE,
)

train_df = df[df["original_circuit"].isin(train_circuits)].copy()
val_df = df[df["original_circuit"].isin(val_circuits)].copy()
test_df = df[df["original_circuit"].isin(test_circuits)].copy()

assert set(train_circuits).isdisjoint(val_circuits)
assert set(train_circuits).isdisjoint(test_circuits)
assert set(val_circuits).isdisjoint(test_circuits)

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train_df), len(val_df), len(test_df)],
        "circuits": [
            train_df["original_circuit"].nunique(),
            val_df["original_circuit"].nunique(),
            test_df["original_circuit"].nunique(),
        ],
        "chains": [
            train_df["chain_name"].nunique(),
            val_df["chain_name"].nunique(),
            test_df["chain_name"].nunique(),
        ],
    }
)
split_summary


## 2. Reward Definition

In [ ]:
train_runtime_99 = float(train_df["total_runtime_seconds"].quantile(0.99))
runtime_denominator = np.log1p(max(train_runtime_99, 1e-9))


def add_reward_columns(frame):
    frame = frame.copy()
    frame["pct_score"] = frame["two_qubit_gate_reduction_pct"] / 100.0
    frame["runtime_score"] = (
        np.log1p(frame["total_runtime_seconds"].clip(lower=0))
        / runtime_denominator
    ).clip(0, 2)
    frame["utility_reward"] = (
        frame["pct_score"] - RUNTIME_WEIGHT * frame["runtime_score"]
    )
    return frame


train_df = add_reward_columns(train_df)
val_df = add_reward_columns(val_df)
test_df = add_reward_columns(test_df)
df = add_reward_columns(df)

reward_summary = pd.DataFrame(
    {
        "train_runtime_99_seconds": [train_runtime_99],
        "runtime_weight": [RUNTIME_WEIGHT],
        "train_utility_mean": [train_df["utility_reward"].mean()],
        "train_utility_min": [train_df["utility_reward"].min()],
        "train_utility_max": [train_df["utility_reward"].max()],
    }
)
reward_summary


eval_frames = {
    "validation": val_df,
    "val": val_df,
    "test": test_df,
}
if EVAL_SPLIT not in eval_frames:
    raise ValueError('EVAL_SPLIT must be "validation", "val", or "test".')

eval_df = eval_frames[EVAL_SPLIT]
eval_split_name = "validation" if EVAL_SPLIT == "val" else EVAL_SPLIT
print(f"Evaluating on {eval_split_name}: {eval_df['original_circuit'].nunique()} circuits")


## 3. Observation Scaling

In [ ]:
def initial_metric_frame(frame):
    return pd.DataFrame(
        {
            "num_qubits": frame["initial_num_qubits"],
            "depth": frame["initial_depth"],
            "two_qubit_gates": frame["initial_two_qubit_gates"],
            "total_gates": frame["initial_total_gates"],
            "two_qubit_depth": frame["initial_two_qubit_depth"],
            "gate_density": frame["initial_gate_density"],
            "two_qubit_ratio": frame["initial_two_qubit_ratio"],
        }
    )


def final_metric_frame(frame):
    final_total = frame["final_total_gates"].replace(0, np.nan)
    final_qubits = frame["final_num_qubits"].replace(0, np.nan)
    return pd.DataFrame(
        {
            "num_qubits": frame["final_num_qubits"],
            "depth": frame["final_depth"],
            "two_qubit_gates": frame["final_two_qubit_gates"],
            "total_gates": frame["final_total_gates"],
            "two_qubit_depth": frame["final_two_qubit_depth"],
            "gate_density": frame["final_total_gates"] / final_qubits,
            "two_qubit_ratio": frame["final_two_qubit_gates"] / final_total,
        }
    ).fillna(0.0)


train_single_df = train_df[~train_df["chain_name"].str.contains("__", regex=False)]
scale_source = pd.concat(
    [initial_metric_frame(train_df), final_metric_frame(train_single_df)],
    ignore_index=True,
)
metric_scale = scale_source.quantile(0.95).replace(0, 1.0).to_dict()
metric_scale


## 4. Load Model and Evaluate PPO

In [ ]:
prediction_env = OptimizerChainEnv(
    eval_df,
    OPTIMIZER_NAMES,
    CATEGORY_NAMES,
    metric_scale,
    shuffle_each_epoch=False,
)
model = PPO.load(CHAIN_OPTIMIZER_DIR / MODEL_PATH.name, env=prediction_env)


def evaluate_policy(model, frame, method="PPO agent"):
    env = OptimizerChainEnv(
        frame,
        OPTIMIZER_NAMES,
        CATEGORY_NAMES,
        metric_scale,
        shuffle_each_epoch=False,
    )
    selected = []
    invalid_actions = 0

    for circuit in sorted(frame["original_circuit"].unique()):
        obs, _ = env.reset(options={"circuit": circuit})
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(int(action))

        if info.get("invalid_action", False):
            invalid_actions += 1
            continue

        if not (terminated or truncated):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(int(action))

        selected_chain = info["selected_chain"]
        row = frame[
            (frame["original_circuit"] == circuit)
            & (frame["chain_name"] == selected_chain)
        ].iloc[0].copy()
        row["method"] = method
        row["selected_chain"] = selected_chain
        row["invalid_action"] = bool(info.get("invalid_action", False))
        selected.append(row)

    result = pd.DataFrame(selected)
    result.attrs["invalid_action_rate"] = invalid_actions / max(frame["original_circuit"].nunique(), 1)
    return result


## 5. Baselines

In [ ]:
def oracle_rows(frame, score_col="utility_reward"):
    return frame.loc[frame.groupby("original_circuit")[score_col].idxmax()].copy()


def fixed_chain_rows(frame, chain_name, method):
    rows = frame[frame["chain_name"] == chain_name].copy()
    rows["method"] = method
    rows["selected_chain"] = chain_name
    rows["invalid_action"] = False
    return rows


def summarize_selected(selected_rows, method, oracle_by_circuit, invalid_action_rate=0.0):
    selected = selected_rows.set_index("original_circuit")
    merged = selected.join(
        oracle_by_circuit[["oracle_utility_reward"]],
        how="inner",
    )
    return {
        "method": method,
        "circuits": len(merged),
        "avg_reward_utility": merged["utility_reward"].mean(),
        "avg_pct_reduction": merged["two_qubit_gate_reduction_pct"].mean(),
        "avg_raw_reduction": merged["two_qubit_gate_reduction"].mean(),
        "avg_runtime_seconds": merged["total_runtime_seconds"].mean(),
        "avg_regret_vs_utility_oracle": (
            merged["oracle_utility_reward"] - merged["utility_reward"]
        ).mean(),
        "invalid_action_rate": invalid_action_rate,
    }


def summarize_random_expected(frame, oracle_by_circuit):
    random_expected = frame.groupby("original_circuit").agg(
        utility_reward=("utility_reward", "mean"),
        two_qubit_gate_reduction_pct=("two_qubit_gate_reduction_pct", "mean"),
        two_qubit_gate_reduction=("two_qubit_gate_reduction", "mean"),
        total_runtime_seconds=("total_runtime_seconds", "mean"),
    )
    merged = random_expected.join(
        oracle_by_circuit[["oracle_utility_reward"]],
        how="inner",
    )
    return {
        "method": "Random chain",
        "circuits": len(merged),
        "avg_reward_utility": merged["utility_reward"].mean(),
        "avg_pct_reduction": merged["two_qubit_gate_reduction_pct"].mean(),
        "avg_raw_reduction": merged["two_qubit_gate_reduction"].mean(),
        "avg_runtime_seconds": merged["total_runtime_seconds"].mean(),
        "avg_regret_vs_utility_oracle": (
            merged["oracle_utility_reward"] - merged["utility_reward"]
        ).mean(),
        "invalid_action_rate": 0.0,
    }


## 6. Split Comparison


In [ ]:
single_chains = sorted([name for name in train_df["chain_name"].unique() if "__" not in name])
#WISQ_BASELINE_CHAIN = "tket__wisq_bqskit"

best_single_chain = (
    train_df[train_df["chain_name"].isin(single_chains)]
    .groupby("chain_name")["utility_reward"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)

best_overall_chain = (
    train_df.groupby("chain_name")["utility_reward"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)

best_single_label = f"Best single optimizer ({best_single_chain})"
best_overall_label = f"Best overall chain ({best_overall_chain})"
#wisq_label = WISQ_BASELINE_CHAIN

oracle_eval = oracle_rows(eval_df, "utility_reward")
oracle_eval["method"] = "Oracle best chain"
oracle_eval["selected_chain"] = oracle_eval["chain_name"]
oracle_eval["invalid_action"] = False
oracle_by_circuit = oracle_eval.set_index("original_circuit")[["utility_reward"]].rename(
    columns={"utility_reward": "oracle_utility_reward"}
)

ppo_selected = evaluate_policy(model, eval_df, method="PPO agent")
#wisq_selected = fixed_chain_rows(eval_df, WISQ_BASELINE_CHAIN, wisq_label)
best_single_selected = fixed_chain_rows(eval_df, best_single_chain, best_single_label)
best_overall_selected = fixed_chain_rows(eval_df, best_overall_chain, best_overall_label)

comparison = pd.DataFrame(
    [
        summarize_random_expected(eval_df, oracle_by_circuit),
        #summarize_selected(wisq_selected, wisq_label, oracle_by_circuit),
        summarize_selected(best_single_selected, best_single_label, oracle_by_circuit),
        summarize_selected(best_overall_selected, best_overall_label, oracle_by_circuit),
        summarize_selected(
            ppo_selected,
            "PPO agent",
            oracle_by_circuit,
            invalid_action_rate=ppo_selected.attrs.get("invalid_action_rate", 0.0),
        ),
        summarize_selected(oracle_eval, "Oracle best chain", oracle_by_circuit),
    ]
)

comparison = comparison.sort_values("avg_reward_utility", ascending=False).reset_index(drop=True)
comparison


In [ ]:
assert eval_df["original_circuit"].nunique() == len(ppo_selected)
assert ppo_selected["selected_chain"].notna().all()
assert ppo_selected["selected_chain"].isin(eval_df["chain_name"].unique()).all()
assert ppo_selected["selected_chain"].str.count("__").le(1).all()
print(f"Evaluation checks passed on {eval_split_name}.")


## 7. Selected Chain Counts

This shows which optimizer chains were selected by PPO and by the baselines that make an actual chain choice. The random baseline is omitted here because it is reported as an expected average over all chains, not as one selected chain per circuit.

In [ ]:
selected_counts_source = pd.concat(
    [
        ppo_selected[["method", "selected_chain", "original_circuit"]],
        wisq_selected[["method", "selected_chain", "original_circuit"]],
        best_single_selected[["method", "selected_chain", "original_circuit"]],
        best_overall_selected[["method", "selected_chain", "original_circuit"]],
        oracle_eval[["method", "selected_chain", "original_circuit"]],
    ],
    ignore_index=True,
)

selected_chain_counts = (
    selected_counts_source.groupby(["method", "selected_chain"])
    .size()
    .reset_index(name="circuits")
    .sort_values(["method", "circuits"], ascending=[True, False])
)

selected_chain_counts


In [ ]:
import textwrap

def prettify_label(name, wrap_width=22):
    name = str(name)

    name = name.replace("__", " --> ")
    name = name.replace("_", " ")

    replacements = {
        "qiskit standard": "Qiskit",
        "qiskit ai": "Qiskit AI",
        "wisq rules": "WISQ Rules",
        "wisq bqskit": "WISQ BQSKit",
        "tket": "TKET",
        "random chain": "Random Chain",
        "best fixed chain": "Best Fixed Chain",
        "best individual": "Best Individual",
        "oracle": "Oracle",
        "rl policy": "RL Policy",
    }

    for old, new in replacements.items():
        name = name.replace(old, new)

    if wrap_width is None:
        return name

    return textwrap.fill(name, width=wrap_width)


selected_chain_grid = selected_chain_counts.pivot_table(
    index="method",
    columns="selected_chain",
    values="circuits",
    fill_value=0,
    aggfunc="sum",
)

# Keep a stable method order while using the actual labels that include chain names.
method_order = [
    "PPO agent",
    wisq_label,
    best_single_label,
    best_overall_label,
    "Oracle best chain",
]
selected_chain_grid = selected_chain_grid.loc[
    [method for method in method_order if method in selected_chain_grid.index],
    selected_chain_grid.sum(axis=0).sort_values(ascending=False).index,
]

selected_chain_grid_pretty = selected_chain_grid.copy()
selected_chain_grid_pretty.index = [
    prettify_label(name, wrap_width=24) for name in selected_chain_grid_pretty.index
]
selected_chain_grid_pretty.columns = [
    prettify_label(name, wrap_width=18) for name in selected_chain_grid_pretty.columns
]

display(selected_chain_grid_pretty)

plt.figure(figsize=(max(12, 0.6 * selected_chain_grid_pretty.shape[1]), 4.5))
sns.heatmap(
    selected_chain_grid_pretty,
    annot=True,
    fmt="d",
    cmap="Blues",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": f"Number of {eval_split_name} circuits"},
)
plt.xlabel("Selected optimizer chain")
plt.ylabel("Method")
plt.title("Selected Optimizer Chain Counts by Method")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### PPO chain usage on the selected evaluation split

In [ ]:
ppo_chain_counts = (
    ppo_selected["selected_chain"]
    .value_counts()
    .rename_axis("selected_chain")
    .reset_index(name="times_selected")
)

ppo_chain_counts["display_chain"] = ppo_chain_counts["selected_chain"].apply(
    lambda name: prettify_label(name, wrap_width=18)
)

plt.figure(figsize=(max(10, 0.6 * len(ppo_chain_counts)), 4.5))
bars = plt.bar(
    ppo_chain_counts["display_chain"],
    ppo_chain_counts["times_selected"],
    color="#4C78A8",
)

for bar, value in zip(bars, ppo_chain_counts["times_selected"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{int(value)}",
        ha="center",
        va="bottom",
    )

plt.xlabel("Selected optimizer chain")
plt.ylabel("Number of PPO selections")
plt.title(f"PPO Chain Usage on {eval_split_name.title()} Split")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. Visual Comparisons

In [ ]:
import matplotlib.pyplot as plt

def horizontal_bar(
    data,
    x,
    xlabel,
    filename,
    title,
    color="#4C78A8",
    fmt="{:.3f}",
    wrap_width=22,
):
    plot_data = data.sort_values(x).copy()

    plot_data["method_pretty"] = plot_data["method"].apply(
        lambda name: prettify_label(name, wrap_width=wrap_width)
    )

    fig_height = max(3.0, 0.55 * len(plot_data))
    fig, ax = plt.subplots(figsize=(4.8, fig_height))
    ax.grid(False)
    ax.xaxis.grid(False)
    ax.yaxis.grid(False)

    bars = ax.barh(plot_data["method_pretty"], plot_data[x], color=color)

    ax.set_title(title, fontsize=11, pad=10)
    ax.set_xlabel(xlabel, fontsize=10, labelpad=8)
    ax.tick_params(axis="x", labelsize=9)
    ax.tick_params(axis="y", labelsize=8)

    max_value = plot_data[x].max()
    min_value = plot_data[x].min()

    value_range = max_value - min_value
    if value_range == 0:
        value_range = abs(max_value) if max_value != 0 else 1

    padding = 0.03 * value_range

    for bar, value in zip(bars, plot_data[x]):
        ax.text(
            bar.get_width() + padding,
            bar.get_y() + bar.get_height() / 2,
            fmt.format(value),
            va="center",
            fontsize=8,
        )

    ax.set_xlim(
        min(0, min_value) - 0.05 * value_range,
        max_value + 0.22 * value_range,
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.subplots_adjust(left=0.42, right=0.96, bottom=0.16, top=0.88)

    fig.savefig(filename, bbox_inches="tight")
    plt.show()
    plt.close(fig)

In [ ]:
horizontal_bar(
    comparison,
    "avg_reward_utility",
    "Average utility reward",
    filename="avg_reward_utility.pdf",
    title="Average Reward Utility by Method",
)

horizontal_bar(
    comparison,
    "avg_pct_reduction",
    "Average two-qubit gate reduction (%)",
    filename="avg_pct_reduction.pdf",
    title="Average Two-Qubit Gate Reduction by Method",
    color="#59A14F",
    fmt="{:.1f}%",
)

horizontal_bar(
    comparison,
    "avg_runtime_seconds",
    "Average runtime (s)",
    filename="avg_runtime_seconds.pdf",
    title="Average Runtime by Method",
    color="#F28E2B",
    fmt="{:.2f}",
)

## 9. Per-Circuit PPO vs tket__wisq_bqskit

In [ ]:
#wisq_eval_rows = eval_df[eval_df["chain_name"] == WISQ_BASELINE_CHAIN].copy()

utility_compare = (
    ppo_selected.set_index("original_circuit")[["selected_chain", "utility_reward"]]
    .rename(
        columns={
            "selected_chain": "ppo_selected_chain",
            "utility_reward": "ppo_utility",
        }
    )
    .join(
        wisq_eval_rows.set_index("original_circuit")[["utility_reward"]]
        .rename(columns={"utility_reward": "tket__wisq_bqskit_utility"}),
        how="inner",
    )
)
utility_compare["ppo_minus_tket__wisq_bqskit"] = (
    utility_compare["ppo_utility"] - utility_compare["tket__wisq_bqskit_utility"]
)
utility_compare = utility_compare.sort_values("ppo_minus_tket__wisq_bqskit")

display(utility_compare)

baseline_chain_label = prettify_label(WISQ_BASELINE_CHAIN, wrap_width=None)

plt.figure(figsize=(14, 4))
plt.bar(
    utility_compare.index,
    utility_compare["ppo_minus_tket__wisq_bqskit"],
    color=np.where(utility_compare["ppo_minus_tket__wisq_bqskit"] >= 0, "#4C78A8", "#E45756"),
)
plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=90)
plt.ylabel(f"PPO utility - {baseline_chain_label} utility")
plt.title(f"PPO Utility Difference vs {baseline_chain_label} on Each {eval_split_name.title()} Circuit")
plt.tight_layout()
plt.show()
